In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

file_path = r'\\wsl.localhost\Ubuntu-24.04\home\shreya\sim\data\raw\drugbank\full_database.xml'
output_file = 'drugbank_flat.csv'

# write header once
columns = [
    'drug_id', 'secondary_ids', 'name', 'type',
    'description', 'indication',
    'state', 'cas',
    'status', 'targets', 'interactions'
]

pd.DataFrame(columns=columns).to_csv(output_file, index=False)

chunk = []
chunk_size = 500  # adjust if needed

context = ET.iterparse(file_path, events=('end',))

for event, elem in context:
    if elem.tag == 'drug':

        # --- IDs ---
        ids = [i.text for i in elem.findall('.//drugbank-id') if i.text]
        primary_id = ids[0] if ids else None
        secondary_ids = '; '.join(ids[1:])

        # --- basic fields ---
        name = elem.findtext('name')
        drug_type = elem.get('type')
        description = elem.findtext('description')
        indication = elem.findtext('indication')
        state = elem.findtext('state')
        cas = elem.findtext('cas-number')

        # --- status ---
        groups = [g.text for g in elem.findall('.//group') if g.text]
        status = '; '.join(groups)

        # --- targets (flattened) ---
        targets = []
        for t in elem.findall('.//target'):
            t_name = t.findtext('name')
            if t_name:
                targets.append(t_name)
        targets = '; '.join(targets)

        # --- interactions ---
        interactions = []
        for i in elem.findall('.//drug-interaction'):
            i_name = i.findtext('name')
            if i_name:
                interactions.append(i_name)
        interactions = '; '.join(interactions)

        # --- append row ---
        chunk.append({
            'drug_id': primary_id,
            'secondary_ids': secondary_ids,
            'name': name,
            'type': drug_type,
            'description': description,
            'indication': indication,
            'state': state,
            'cas': cas,
            'status': status,
            'targets': targets,
            'interactions': interactions
        })

        # --- write in chunks ---
        if len(chunk) >= chunk_size:
            pd.DataFrame(chunk).to_csv(output_file, mode='a', header=False, index=False)
            chunk = []

        elem.clear()  # 🔥 critical for memory

# write remaining
if chunk:
    pd.DataFrame(chunk).to_csv(output_file, mode='a', header=False, index=False)

print("✅ Done! File saved as drugbank_flat.csv")